In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json 
from pyprojroot import here
import sys
from tqdm import tqdm
sys.path.append(str(here() / 'methods'))


#my functions
from additional_utils.functions import coordinates2plot, mini_ranking_evaluations

processed_data = here() / 'data' / 'processed_data'
cv_results_data = here() / 'data' / 'processed_data' / 'transductive_data' / 'results'


In [ ]:
#This are the results of the machine learning models

files = [f.name for f in cv_results_data.iterdir() if f.is_file()]

print(files)

pred_prob_dict = []
for file in tqdm(files, desc='Files'):
      with open(cv_results_data / file, 'r') as f:
         d = json.load(f)
         pred_prob_dict = pred_prob_dict + d

df = pd.DataFrame(pred_prob_dict)[[
      'subset', 
      'model', 
      'y_test_CSU', 
      'calibrated_probabilities_CSU',
      'y_test_CSFull', 
      'calibrated_probabilities_CSFull']]


In [ ]:
#This dataset is simply adding the CRI as a ranking system

df_cri = pd.read_feather(processed_data / 'cv_CRI.feather',
                         columns= [
                            #'param_id', 
                            'subset', 
                            'model', 
                            'y_test_CSFull', 
                            'calibrated_probabilities_CSFull',
                            ]
                         )

df = pd.concat([df,df_cri]).reset_index(drop=True)

In [ ]:
#replace model names
df['model'] = df['model'].replace({
    'hdsrf': 'HDSRF',
    'pubagging': 'PUBagging',
    'cri': 'Ranking by CRI*'
})

In [ ]:
#replace columns
df.columns = df.columns.str.replace('calibrated', 'cal', regex=False)


# Supplementary Information H1

### Supplementary Information H1 Table 9

In [ ]:
#**********************************************************
#What to plot #CSFull, CSU, YSFull
type_testset = 'CSFull'
y_test_col = f'y_test_{type_testset}'
y_prob_col = f'cal_probabilities_{type_testset}' 
thresholds2measure = list(np.round(np.arange(0.025,1.025, 0.025),3))

if type_testset == 'CSU':
    df = df[df['model'] != 'Ranking by CRI*'].reset_index()

topkmetrics_df = coordinates2plot(
        df2work = df, #df2work
        y_test_col = y_test_col,
        pred_prob_col = y_prob_col,
        ks = thresholds2measure, #thresholds to measure
        group_col = 'model',
        )

In [ ]:
grouped_df = topkmetrics_df.groupby(['model', 'topk_instances']).agg(
    #correct measures
    mean_robust_recall = ('robust_recall', 'mean'),
    std_robust_recall = ('robust_recall', 'std'),
    mean_robust_precision = ('robust_precision', 'mean'),
    std_robust_precision = ('robust_precision', 'std'),
    mean_robust_lift = ('robust_lift', 'mean'),
    std_robust_lift = ('robust_lift', 'std'),
    #biased measures
    mean_biased_recall = ('biased_recall', 'mean'),
    std_biased_recall = ('biased_recall', 'std'),
    mean_biased_precision = ('biased_precision', 'mean'),
    std_biased_precision = ('biased_precision', 'std'),
    mean_biased_lift = ('biased_lift', 'mean'),
    std_biased_lift = ('biased_lift', 'std'),
    #null models
    avg_prevalence = ('prevalence', 'mean'),
    mean_null_recall = ('null_recall', 'mean')
)
grouped_df = grouped_df.reset_index()


In [ ]:
#Supplementary Information H1 Table 9

cols = [
    "model", "topk_instances",
    "mean_robust_recall", "std_robust_recall",
    "mean_robust_precision", "std_robust_precision",
    "mean_robust_lift", "std_robust_lift"
]

df_sub = grouped_df[cols].copy()
def fmt(mean, std):
    return mean.round(3).astype(str) + " (" + std.round(3).astype(str) + ")"

df_sub["robust_recall"] = fmt(df_sub["mean_robust_recall"], df_sub["std_robust_recall"])
df_sub["robust_precision"] = fmt(df_sub["mean_robust_precision"], df_sub["std_robust_precision"])
df_sub["robust_lift"] = fmt(df_sub["mean_robust_lift"], df_sub["std_robust_lift"])
df_sub = df_sub[["model", "topk_instances",
                 "robust_recall", "robust_precision", "robust_lift"]]
df_pivot = df_sub.pivot(index="topk_instances", columns="model")
df_pivot.columns = [
    f"{metric}_{model}"
    for metric, model in df_pivot.columns
]

df_pivot = df_pivot#.reset_index()
print(df_pivot)

# Main Text Figure 2

In [ ]:
import matplotlib.pyplot as plt

#******************************
# Loops

algorithms = [ 'PUBagging', 'HDSRF', 'Ranking by CRI*']
measures = ['recall', 'precision', 'lift']
type_measure = 'robust'
figsize = (10,5)

if type_testset in ['CSU']:
    algorithms = [ 'PUBagging', 'HDSRF']
    figsize = (7.5,5)

if type_testset in ['YSFull']:
    algorithms = [ 'PUBagging-EPN', 'HDSRF-EPN']
    figsize = (7.5,5)


#******************************
#Aestetics

plt.style.use('ggplot')
lower_blue = '#40798C'
blue = '#2b9eb3'

#******************************
#Notes
notes = topkmetrics_df[['model', f'{type_measure}_average_gain', f'{type_measure}_average_precision', f'{type_measure}_average_lift']].drop_duplicates().groupby('model').agg(('mean', 'std')).round(2).to_dict()
#Label position
maxis = {
    'recall':(0.7, 0.25),
    'precision':(0.02, 0.05),
    'lift': (0.02, 2)
}
#******************************
#Axis labels
if type_measure == 'robust':
    measures_labels = [
        r'$R_{R}@k$',
        r'$P_{R}@k$',
        r'$L_{R}@k$'
    ]
else:
    measures_labels = [
        r'$R@k$',
        r'$P@k$',
        r'$L@k$'
    ]
measures_labels_dict = dict(zip(measures, measures_labels))
#******************************
fig, axs = plt.subplots(
    3,len(algorithms),
    gridspec_kw={'height_ratios': [2, 1, 1]},
    figsize=figsize)

for col, algorithm in enumerate(algorithms):

    df2plot = grouped_df[grouped_df['model'] == algorithm]

    for row, measure in enumerate(measures):

        pointsize = 3 if row == 0 else 6
        if row == 0:
            avg_n = notes[(f'{type_measure}_average_gain', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_gain', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{Gain}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 1:
            avg_n = notes[(f'{type_measure}_average_precision', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_precision', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{P_R}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 2:
            avg_n = notes[(f'{type_measure}_average_lift', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_lift', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{L_R}$: ' + f'\n {avg_n}  ±  {std_n}'

        ax = axs[row, col]

        ax.errorbar(
            x=df2plot['topk_instances'],
            y=df2plot[f'mean_{type_measure}_{measure}'],
            color=lower_blue,
            label = 'Our model',
            marker = 'o',
            markersize = pointsize,
            
        )

        if row != 1:
            ax.text(
                maxis[measure][0],
                maxis[measure][1],
                textonplot,
                fontsize=6,
                verticalalignment='bottom'
                )
            #print(textonplot)

        if measure == 'recall':
            ax.fill_between(
                x = df2plot['topk_instances'],
                y1 = df2plot['mean_null_recall'],
                y2 = df2plot[f'mean_{type_measure}_{measure}'],
                #step='mid',
                alpha=0.8,
                color=blue,
                label='Gain',
                linestyle = '-'
            )


            ax.plot(
                df2plot['topk_instances'],
                df2plot['mean_null_recall'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )

        elif measure == 'precision':
            ax.plot(
                df2plot['topk_instances'],
                df2plot['avg_prevalence'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-0.05, 0.25)
            ax.set_xlim(0, 0.275)


        
        elif measure == 'lift':
            ax.plot(
                df2plot['topk_instances'],
                np.repeat([1], len(df2plot['topk_instances'])),
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-1, 8)
            ax.set_xlim(0, 0.275)


        ax.tick_params(axis='x', rotation=0, labelsize = 6)
        ax.tick_params(axis='y', rotation=0, labelsize = 6)


        if col == 0:
            ax.set_ylabel(measures_labels_dict[measure], fontsize = 9)
            if row == 0:
                ax.legend(loc='upper left', fontsize=8)
        else:
            ax.set_yticklabels('')

        if row == 0:
            ax.set_title(algorithm, fontsize = 9)

        if row == 1:
            ax.set_xticklabels('')
        if (row == 2) & ( col == 1 ):
            ax.set_xlabel('Top k instances', fontsize = 9)
            #
        
# reduce whitespace between plots
fig.subplots_adjust(hspace=0.2, wspace=0.05)    

#plt.tight_layout()
#filename = output_folder / f'evaluation_{type_measure}_{type_testset}.png'
#plt.savefig(filename, dpi=300, bbox_inches="tight")
plt.show()

# Supplementary Information H2

In [ ]:
#CSFull
average_gain_unCSFull_list = []
average_gain_calCSFull_list = []
average_lift_unCSFull_list = []
average_lift_calCSFull_list = []

df = df[df['model'] != 'Ranking by CRI*'].reset_index(drop=True)

for row in tqdm(range(len(df)), desc = 'Row'):
    ### CSFull
    y_true_CSFull = df.iloc[row]['y_test_CSFull']
    unCSFull_prob = df.iloc[row]['un_probabilities_CSFull']
    calCSFull_prob = df.iloc[row]['cal_probabilities_CSFull']
    ### Uncalibrated CSFull
    average_gain_unCSFull, average_lift_unCSFull = mini_ranking_evaluations(
        y_true = y_true_CSFull,
        y_pred_prob = unCSFull_prob, #PREDPROB
        top_k = len(y_true_CSFull), 
        prevalence = None)

    average_gain_unCSFull_list.append(average_gain_unCSFull) #Saving
    average_lift_unCSFull_list.append(average_lift_unCSFull) #Saving


    ### Calibrated CSFull
    average_gain_calCSFull, average_lift_calCSFull = mini_ranking_evaluations(
        y_true = y_true_CSFull,
        y_pred_prob = calCSFull_prob, #PREDPROB
        top_k = len(y_true_CSFull), 
        prevalence = None)

    average_gain_calCSFull_list.append(average_gain_calCSFull) #Saving
    average_lift_calCSFull_list.append(average_lift_calCSFull) #Saving


#Include in dataset

#CSFull
df['avg_gain_unCSFull'] = average_gain_unCSFull_list
df['avg_gain_calCSFull'] = average_gain_calCSFull_list
df['avg_lift_unCSFull'] = average_lift_unCSFull_list
df['avg_lift_calCSFull'] = average_lift_calCSFull_list



In [ ]:
#figures_folder = here() / 'figures' 

import matplotlib.pyplot as plt
import numpy as np

test_type = 'CSFull'
probability_type = 'cal'      # calibrated
exclude_probability_type = 'un'  # uncalibrated

plt.style.use('ggplot')


# keep only probability columns for the selected test type
cols2keep = [col for col in df.columns if test_type in col]
cols2keep = [col for col in cols2keep if exclude_probability_type not in col]

models = ['HDSRF', 'PUBagging']
subsets = ["A", "B", "C", "D"]

# ---- Create a single figure with 2 rows and 4 columns ----
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharey='row')

for row_idx, model in enumerate(models):

    # filter the df for this model
    df2plot = df[cols2keep + ['model']]
    df2plot = df2plot[df2plot['model'] == model].reset_index(drop=True)

    y_test_col = 'y_test_' + test_type
    y_prob_col = probability_type + '_probabilities_' + test_type

    for col_idx, subset in enumerate(subsets):

        ax = axes[row_idx][col_idx]

        # Extract arrays
        y_test_array = np.array(df2plot.iloc[col_idx][y_test_col])
        pred_prob_array = np.array(df2plot.iloc[col_idx][y_prob_col])

        avg_gain_i = round(df2plot.iloc[col_idx]['avg_gain_' + probability_type + test_type], 3)
        avg_lift_i = round(df2plot.iloc[col_idx]['avg_lift_' + probability_type + test_type], 3)
        

        # Separate by label
        pred_prob_1 = pred_prob_array[y_test_array == 1]
        pred_prob_0 = pred_prob_array[y_test_array == 0]

        # ----- Plot histograms -----
        ax.hist(pred_prob_0, bins=100, alpha=0.5, label='Negative', color='blue', density=True)
        ax.hist(pred_prob_1, bins=100, alpha=0.5, label='Positive', color='red', density=True)

        # ----- Text -----
        x4text = 0.10
        y4text = 0.90
        ax.text(x4text, y4text, f'Avg. gain: {avg_gain_i}', transform=ax.transAxes, fontsize=12)
        ax.text(x4text, y4text - 0.07, f'Avg. lift: {avg_lift_i}', transform=ax.transAxes, fontsize=12)
        # ax.text(x4text, y4text - 0.14, f'MAP: {map_i}', transform=ax.transAxes, fontsize=9)

        # ----- Titles -----
        if row_idx == 0:  # Only on top row
            ax.set_title(f'Fold {subset}', fontsize=11)
        else:
            ax.set_title("")  # remove bottom titles

        # ----- X-labels -----
        if row_idx == 1:   # bottom row only
            ax.set_xlabel('Predicted Probability')
        else:
            ax.set_xlabel("")

        # ----- Y-label only for first column -----
        if col_idx == 0:
            ax.set_ylabel(f'Density ({model})', fontsize=11)
        else:
            ax.set_ylabel("")

        if (col_idx == 3) & (row_idx == 0):
            ax.legend()

fig.tight_layout()

# Save the figure
#filename = 'probabilities_distribution.png'
#plt.savefig(figures_folder / filename, dpi=300, bbox_inches='tight', facecolor='white')

plt.show()

# Supplementary Information H3

In [ ]:
import matplotlib.pyplot as plt


#******************************
# Loops

algorithms = [ 'PUBagging', 'HDSRF', 'Ranking by CRI*']
measures = ['recall', 'precision', 'lift']
type_measure = 'biased'
figsize = (10,5)

if type_testset == 'CSU':
    algorithms = [ 'PUBagging', 'HDSRF']
    figsize = (7.5,5)
    

#******************************
#Aestetics

plt.style.use('ggplot')
lower_blue = '#40798C'
blue = '#2b9eb3'

#******************************
#Notes
notes = topkmetrics_df[['model', f'{type_measure}_average_gain', f'{type_measure}_average_precision', f'{type_measure}_average_lift']].drop_duplicates().groupby('model').agg(('mean', 'std')).round(2).to_dict()
#Label position
maxis = {
    'recall':(0.7, 0.25),
    'precision':(0.02, 0.05),
    'lift': (0.2, 6)
}
#******************************
#Axis labels
if type_measure == 'robust':
    measures_labels = [
        r'$R_{R}@k$',
        r'$P_{R}@k$',
        r'$L_{R}@k$'
    ]
else:
    measures_labels = [
        r'$R@k$',
        r'$P@k$',
        r'$L@k$'
    ]
measures_labels_dict = dict(zip(measures, measures_labels))
#******************************
fig, axs = plt.subplots(
    3,len(algorithms),
    gridspec_kw={'height_ratios': [2, 1, 1]},
    figsize=figsize)

for col, algorithm in enumerate(algorithms):

    df2plot = grouped_df[grouped_df['model'] == algorithm]

    for row, measure in enumerate(measures):

        pointsize = 3 if row == 0 else 6
        if row == 0:
            avg_n = notes[(f'{type_measure}_average_gain', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_gain', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{Gain}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 1:
            avg_n = notes[(f'{type_measure}_average_precision', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_precision', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{P_R}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 2:
            avg_n = notes[(f'{type_measure}_average_lift', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_lift', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{L_R}$: ' + f'\n {avg_n}  ±  {std_n}'

        ax = axs[row, col]

        ax.errorbar(
            x=df2plot['topk_instances'],
            y=df2plot[f'mean_{type_measure}_{measure}'],
            color=lower_blue,
            label = 'Our model',
            marker = 'o',
            markersize = pointsize,
            
        )

        if row != 1:
            ax.text(
                maxis[measure][0],
                maxis[measure][1],
                textonplot,
                fontsize=6,
                verticalalignment='bottom'
                )
            #print(textonplot)

        if measure == 'recall':
            ax.fill_between(
                x = df2plot['topk_instances'],
                y1 = df2plot['mean_null_recall'],
                y2 = df2plot[f'mean_{type_measure}_{measure}'],
                #step='mid',
                alpha=0.8,
                color=blue,
                label='Gain',
                linestyle = '-'
            )


            ax.plot(
                df2plot['topk_instances'],
                df2plot['mean_null_recall'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )

        elif measure == 'precision':
            ax.plot(
                df2plot['topk_instances'],
                df2plot['avg_prevalence'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-0.05, 0.35)
            ax.set_xlim(0, 0.275)


        
        elif measure == 'lift':
            ax.plot(
                df2plot['topk_instances'],
                np.repeat([1], len(df2plot['topk_instances'])),
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-1, 10)
            ax.set_xlim(0, 0.275)


        ax.tick_params(axis='x', rotation=0, labelsize = 6)
        ax.tick_params(axis='y', rotation=0, labelsize = 6)


        if col == 0:
            ax.set_ylabel(measures_labels_dict[measure], fontsize = 9)
            if row == 0:
                ax.legend(loc='upper left', fontsize=8)
        else:
            ax.set_yticklabels('')

        if row == 0:
            ax.set_title(algorithm, fontsize = 9)

        if row == 1:
            ax.set_xticklabels('')
        if (row == 2) & ( col == 1 ):
            ax.set_xlabel('Top k instances', fontsize = 9)
            #
        
# reduce whitespace between plots
fig.subplots_adjust(hspace=0.2, wspace=0.05)    

plt.tight_layout()

plt.show()

In [ ]:
#Supplementary Information H3. Table 10

cols = [
    "model", "topk_instances",
    "mean_biased_recall", "std_biased_recall",
    "mean_biased_precision", "std_biased_precision",
    "mean_biased_lift", "std_biased_lift"
]

df_sub = grouped_df[cols].copy()
def fmt(mean, std):
    return mean.round(3).astype(str) + " (" + std.round(3).astype(str) + ")"

df_sub["biased_recall"] = fmt(df_sub["mean_biased_recall"], df_sub["std_biased_recall"])
df_sub["biased_precision"] = fmt(df_sub["mean_biased_precision"], df_sub["std_biased_precision"])
df_sub["biased_lift"] = fmt(df_sub["mean_biased_lift"], df_sub["std_biased_lift"])
df_sub = df_sub[["model", "topk_instances",
                 "biased_recall", "biased_precision", "biased_lift"]]
df_pivot = df_sub.pivot(index="topk_instances", columns="model")
df_pivot.columns = [
    f"{metric}_{model}"
    for metric, model in df_pivot.columns
]

df_pivot = df_pivot
print(df_pivot)

# Supplementary Information H4

In [ ]:
#**********************************************************
#What to plot #CSFull, CSU, YSFull
type_testset = 'CSU'
y_test_col = f'y_test_{type_testset}'
y_prob_col = f'cal_probabilities_{type_testset}' 
thresholds2measure = list(np.round(np.arange(0.025,1.025, 0.025),3))

if type_testset == 'CSU':
    df = df[df['model'] != 'Ranking by CRI*'].reset_index()

topkmetrics_df = coordinates2plot(
        df2work = df, #df2work
        y_test_col = y_test_col,
        pred_prob_col = y_prob_col,
        ks = thresholds2measure, #thresholds to measure
        group_col = 'model',
        )

In [ ]:
grouped_df = topkmetrics_df.groupby(['model', 'topk_instances']).agg(
    #correct measures
    mean_robust_recall = ('robust_recall', 'mean'),
    std_robust_recall = ('robust_recall', 'std'),
    mean_robust_precision = ('robust_precision', 'mean'),
    std_robust_precision = ('robust_precision', 'std'),
    mean_robust_lift = ('robust_lift', 'mean'),
    std_robust_lift = ('robust_lift', 'std'),
    #biased measures
    mean_biased_recall = ('biased_recall', 'mean'),
    std_biased_recall = ('biased_recall', 'std'),
    mean_biased_precision = ('biased_precision', 'mean'),
    std_biased_precision = ('biased_precision', 'std'),
    mean_biased_lift = ('biased_lift', 'mean'),
    std_biased_lift = ('biased_lift', 'std'),
    #null models
    avg_prevalence = ('prevalence', 'mean'),
    mean_null_recall = ('null_recall', 'mean')
)
grouped_df = grouped_df.reset_index()


In [ ]:
import matplotlib.pyplot as plt

#******************************
# Loops

algorithms = [ 'PUBagging', 'HDSRF', 'Ranking by CRI*']
measures = ['recall', 'precision', 'lift']
type_measure = 'robust'
figsize = (10,5)

if type_testset == 'CSU':
    algorithms = [ 'PUBagging', 'HDSRF']
    figsize = (7.5,5)
    

#******************************
#Aestetics

plt.style.use('ggplot')
lower_blue = '#40798C'
blue = '#2b9eb3'

#******************************
#Notes
notes = topkmetrics_df[['model', f'{type_measure}_average_gain', f'{type_measure}_average_precision', f'{type_measure}_average_lift']].drop_duplicates().groupby('model').agg(('mean', 'std')).round(2).to_dict()
#Label position
maxis = {
    'recall':(0.7, 0.25),
    'precision':(0.02, 0.05),
    'lift': (0.02, 2)
}
#******************************
#Axis labels
if type_measure == 'robust':
    measures_labels = [
        r'$R_{R}@k$',
        r'$P_{R}@k$',
        r'$L_{R}@k$'
    ]
else:
    measures_labels = [
        r'$R@k$',
        r'$P@k$',
        r'$L@k$'
    ]
measures_labels_dict = dict(zip(measures, measures_labels))
#******************************
fig, axs = plt.subplots(
    3,len(algorithms),
    gridspec_kw={'height_ratios': [2, 1, 1]},
    figsize=figsize)

for col, algorithm in enumerate(algorithms):

    df2plot = grouped_df[grouped_df['model'] == algorithm]

    for row, measure in enumerate(measures):

        pointsize = 3 if row == 0 else 6
        if row == 0:
            avg_n = notes[(f'{type_measure}_average_gain', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_gain', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{Gain}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 1:
            avg_n = notes[(f'{type_measure}_average_precision', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_precision', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{P_R}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 2:
            avg_n = notes[(f'{type_measure}_average_lift', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_lift', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{L_R}$: ' + f'\n {avg_n}  ±  {std_n}'

        ax = axs[row, col]

        ax.errorbar(
            x=df2plot['topk_instances'],
            y=df2plot[f'mean_{type_measure}_{measure}'],
            color=lower_blue,
            label = 'Our model',
            marker = 'o',
            markersize = pointsize,
            
        )

        if row != 1:
            ax.text(
                maxis[measure][0],
                maxis[measure][1],
                textonplot,
                fontsize=6,
                verticalalignment='bottom'
                )
            #print(textonplot)

        if measure == 'recall':
            ax.fill_between(
                x = df2plot['topk_instances'],
                y1 = df2plot['mean_null_recall'],
                y2 = df2plot[f'mean_{type_measure}_{measure}'],
                #step='mid',
                alpha=0.8,
                color=blue,
                label='Gain',
                linestyle = '-'
            )


            ax.plot(
                df2plot['topk_instances'],
                df2plot['mean_null_recall'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )

        elif measure == 'precision':
            ax.plot(
                df2plot['topk_instances'],
                df2plot['avg_prevalence'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-0.05, 0.25)
            ax.set_xlim(0, 0.275)


        
        elif measure == 'lift':
            ax.plot(
                df2plot['topk_instances'],
                np.repeat([1], len(df2plot['topk_instances'])),
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-1, 8)
            ax.set_xlim(0, 0.275)


        ax.tick_params(axis='x', rotation=0, labelsize = 6)
        ax.tick_params(axis='y', rotation=0, labelsize = 6)


        if col == 0:
            ax.set_ylabel(measures_labels_dict[measure], fontsize = 9)
            if row == 0:
                ax.legend(loc='upper left', fontsize=8)
        else:
            ax.set_yticklabels('')

        if row == 0:
            ax.set_title(algorithm, fontsize = 9)

        if row == 1:
            ax.set_xticklabels('')
        if (row == 2) & ( col == 1 ):
            ax.set_xlabel('Top k instances', fontsize = 9)
            #
        
# reduce whitespace between plots
fig.subplots_adjust(hspace=0.2, wspace=0.05)    

plt.show()